In [1]:
from tqdm import tqdm
import shutil

def rearrange_directories(root_path, subset):
    source_rgb = os.path.join(root_path, 'rgb', 'clevr_complex', subset)
    source_depth = os.path.join(root_path, 'depth_euclidean', 'clevr_complex', subset)

    destination_root = os.path.join(root_path, 'clevr_complex', subset)
    destination_rgb = os.path.join(destination_root, 'rgb')
    destination_depth = os.path.join(destination_root, 'depth_euclidean')

    # Create destination directories if they don't exist
    os.makedirs(destination_root, exist_ok=True)
    os.makedirs(destination_rgb, exist_ok=True)
    os.makedirs(destination_depth, exist_ok=True)

    # Move and rename directories
    shutil.move(source_rgb, destination_rgb)
    shutil.move(source_depth, destination_depth)

# root = './data'
# subset = 'test'
# rearrange_directories(root, subset)


def rename(root_dir):
    import os
    import re

    
    splits = ["train", "val", "test"]
    domains = ["semantic"]

    for split in splits:
        for domain in domains:
            dir_path = os.path.join(root_dir, split, domain)

            for filename in tqdm(os.listdir(dir_path)):
                match = re.match(r'point_(\d+)_view_(\d+)_domain_(.+)\.png', filename)
                if match:
                    new_name = f"{match.group(1)}_{match.group(2)}.png"
                    old_path = os.path.join(dir_path, filename)
                    new_path = os.path.join(dir_path, new_name)

                    os.rename(old_path, new_path)

    print("Files renamed successfully.")


In [3]:

root_dir = '/home/data/dq/clevr_complex'
rename(root_dir)

100%|██████████| 50000/50000 [00:00<00:00, 1023135.73it/s]


100%|██████████| 5000/5000 [01:42<00:00, 48.55it/s]

Files renamed successfully.


In [8]:
from PIL import Image
import numpy as np
import os
from tqdm import tqdm

def image_stats(image_path):
    """Read an image and return its statistics (mean and squared values)."""
    image = Image.open(image_path).convert('RGB')
    image_np = np.array(image) #/ 255.0  # Normalize pixel values to [0, 1]
    mean = np.mean(image_np, axis=(0, 1))  # Channel-wise mean
    std = np.std(image_np, axis=(0, 1))  # Squared mean for std calculation
    return mean, std#, image_np.size / 3  # Return the mean, squared mean, and number of pixels (per channel)

def update_running_stats(stats, new_data):
    """Update running means and squared means given new data."""
    (running_mean, running_sq_mean, total_pixels), (mean, sq_mean, pixels) = stats, new_data
    total_pixels += pixels
    running_mean = (running_mean * (total_pixels - pixels) + mean * pixels) / total_pixels
    running_sq_mean = (running_sq_mean * (total_pixels - pixels) + sq_mean * pixels) / total_pixels
    return running_mean, running_sq_mean, total_pixels

def calculate_overall_stats(directory):
    total_mean = np.zeros(3)
    total_std = np.zeros(3)
    total_objects = 0

    for filename in tqdm(os.listdir(directory)):
        if filename.endswith(('.png', '.jpg', '.jpeg')):  # Add other file extensions if needed.
            filepath = os.path.join(directory, filename)
            mean, std = image_stats(filepath)
            total_mean += mean 
            total_std += std 
            total_objects += 1


    return total_mean / total_objects, total_std / total_objects


In [6]:
image_stats('/home/data/dq/clevr_complex/train/rgb/1002_0.png')

(array([123.43654633, 120.06177902, 117.49253845]),
 array([26.15464732, 27.19059932, 27.6066423 ]))

In [9]:

directory = '/home/data/dq/clevr_complex/val/rgb'  # Update this path
mean, std = calculate_overall_stats(directory)

print(f'Mean: {mean}')
print(f'Standard Deviation: {std}')


  0%|          | 0/5000 [00:00<?, ?it/s]

100%|██████████| 5000/5000 [05:06<00:00, 16.31it/s]

Mean: [121.26299414 120.22486951 118.50290991]
Standard Deviation: [23.04689685 22.30750676 22.16528343]


In [15]:

directory = '/home/data/dq/clevr_complex/train/rgb'  # Update this path
mean, std = calculate_overall_stats(directory)

print(f'Mean: {mean}')
print(f'Standard Deviation: {std}')


 72%|███████▏  | 35792/50000 [43:02<4:55:58,  1.25s/it] 